# Shakespeare CSV Analysis

This notebook performs an exploratory data analysis (EDA) of `Shakespeare_data.csv` using pandas.

It covers:
- dataset shape and schema
- missing values and uniqueness
- categorical and numeric summaries
- text-length analysis for line content
- play-level and speaker-level aggregates

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

csv_path = Path('Shakespeare_data.csv')
if not csv_path.exists():
    csv_path = Path('dataset') / 'Shakespeare_data.csv'

df = pd.read_csv(csv_path)
print(f'Loaded: {csv_path.resolve()}')
print(f'Shape: {df.shape}')

Loaded: /home/pdutta/bart-trans/dataset/Shakespeare_data.csv
Shape: (111396, 6)


In [2]:
# Quick structural overview
display(df.head(10))
display(df.sample(10, random_state=42))

info_df = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(t) for t in df.dtypes],
    'non_null_count': df.notna().sum().values,
    'null_count': df.isna().sum().values,
    'null_pct': (df.isna().mean() * 100).round(2).values,
    'n_unique': df.nunique(dropna=True).values
}).sort_values('null_pct', ascending=False)

display(info_df)

,Dataline,Play,PlayerLinenumber,ActSceneLine,Player,PlayerLine
0,1,Henry IV,NaN,NaN,NaN,ACT I
1,2,Henry IV,NaN,NaN,NaN,SCENE I. London. The palace.
2,3,Henry IV,NaN,NaN,NaN,"Enter KING HENRY, LORD JOHN OF LANCASTER, the ..."
3,4,Henry IV,1.0,1.1.1,KING HENRY IV,"So shaken as we are, so wan with care,"
4,5,Henry IV,1.0,1.1.2,KING HENRY IV,"Find we a time for frighted peace to pant,"
5,6,Henry IV,1.0,1.1.3,KING HENRY IV,And breathe short-winded accents of new broils
6,7,Henry IV,1.0,1.1.4,KING HENRY IV,To be commenced in strands afar remote.
7,8,Henry IV,1.0,1.1.5,KING HENRY IV,No more the thirsty entrance of this soil
8,9,Henry IV,1.0,1.1.6,KING HENRY IV,Shall daub her lips with her own children's bl...
9,10,Henry IV,1.0,1.1.7,KING HENRY IV,"Nor more shall trenching war channel her fields,"


,Dataline,Play,PlayerLinenumber,ActSceneLine,Player,PlayerLine
49287,49288,King Lear,62.0,1.1.249,CORDELIA,"That hath deprived me of your grace and favour,"
76725,76726,Pericles,1.0,2.4.10,HELICANUS,"Their bodies, even to loathing, for they so st..."
46485,46486,Julius Caesar,38.0,1.2.145,CASSIUS,Men at some time are masters of their fates:
76934,76935,Pericles,37.0,3.5.147,GOWER,"Disgorges such a tempest forth,"
35083,35084,Hamlet,50.0,3.4.177,HAMLET,"That monster, custom, who all sense doth eat,"
51366,51367,King Lear,15.0,3.7.37,CORNWALL,"To this chair bind him. Villain, thou shalt fi..."
102689,102690,Troilus and Cressida,4.0,5.5.28,NESTOR,Dexterity so obeying appetite
99928,99929,Troilus and Cressida,31.0,1.3.279,AENEAS,"Hector, in view of Trojans and of Greeks,"
75859,75860,Pericles,8.0,1.1.48,PERICLES,"Who know the world, see heaven, but, feeling woe,"
105366,105367,Twelfth Night,88.0,5.1.243,SEBASTIAN,"I should my tears let fall upon your cheek,"


,column,dtype,non_null_count,null_count,null_pct,n_unique
3,ActSceneLine,object,105153,6243,5.60,16122
4,Player,object,111389,7,0.01,934
1,Play,object,111396,0,0.00,36
0,Dataline,int64,111396,0,0.00,111396
2,PlayerLinenumber,float64,111393,3,0.00,405
5,PlayerLine,object,111396,0,0.00,107580


In [3]:
# Basic missing-data summary
missing_summary = (
    df.isna()
      .sum()
      .rename('null_count')
      .to_frame()
      .assign(null_pct=lambda x: (x['null_count'] / len(df) * 100).round(2))
      .sort_values('null_pct', ascending=False)
)
display(missing_summary)

,null_count,null_pct
ActSceneLine,6243,5.60
Player,7,0.01
Play,0,0.00
Dataline,0,0.00
PlayerLinenumber,3,0.00
PlayerLine,0,0.00


In [4]:
# Categorical summaries (object/string columns)
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Object columns:', obj_cols)

for col in obj_cols:
    print('\n' + '=' * 90)
    print(f'Column: {col}')
    top = df[col].value_counts(dropna=False).head(10)
    display(top.to_frame('count'))

Object columns: ['Play', 'ActSceneLine', 'Player', 'PlayerLine']

Column: Play


,count
Play,
Hamlet,4244
Coriolanus,3992
Cymbeline,3958
Richard III,3941
Antony and Cleopatra,3862
King Lear,3766
Othello,3762
Troilus and Cressida,3711
A Winters Tale,3489



Column: ActSceneLine


,count
ActSceneLine,
NaN,6243
5.1.3,37
2.1.10,36
1.1.8,36
1.1.1,36
1.1.11,36
1.1.10,36
1.1.13,36
1.1.9,36



Column: Player


,count
Player,
GLOUCESTER,1920
HAMLET,1582
IAGO,1161
FALSTAFF,1117
KING HENRY V,1086
BRUTUS,1051
OTHELLO,928
MARK ANTONY,927
KING HENRY VI,917



Column: PlayerLine


,count
PlayerLine,
Exit,564
Exeunt,550
ANTIPHOLUS,175
DON,101
Aside,50
Enter a Messenger,47
ACT III,36
ACT IV,36
ACT V,36


In [5]:
# Numeric summaries, if numeric columns exist
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print('Numeric columns:', num_cols)

if num_cols:
    display(df[num_cols].describe().T)
else:
    print('No numeric columns detected by pandas.')

Numeric columns: ['Dataline', 'PlayerLinenumber']


,count,mean,std,min,25%,50%,75%,max
Dataline,111396.0,55698.500000,32157.399631,1.0,27849.75,55698.5,83547.25,111396.0
PlayerLinenumber,111393.0,36.885334,39.985840,1.0,10.00,25.0,50.00,405.0


In [6]:
# Text-length analysis for likely text columns
likely_text_cols = [c for c in ['PlayerLine', 'Play', 'Player', 'ActSceneLine'] if c in df.columns]
print('Analyzed text columns:', likely_text_cols)

text_len_stats = []
for col in likely_text_cols:
    lengths = df[col].astype(str).str.len()
    text_len_stats.append({
        'column': col,
        'min_len': lengths.min(),
        'median_len': lengths.median(),
        'mean_len': round(lengths.mean(), 2),
        'p95_len': lengths.quantile(0.95),
        'max_len': lengths.max()
    })

display(pd.DataFrame(text_len_stats).sort_values('mean_len', ascending=False))

if 'PlayerLine' in df.columns:
    tmp = df.assign(PlayerLine_len=df['PlayerLine'].astype(str).str.len())
    print('Top 10 longest PlayerLine entries:')
    display(tmp.nlargest(10, 'PlayerLine_len')[['Play', 'Player', 'ActSceneLine', 'PlayerLine_len', 'PlayerLine']])

Analyzed text columns: ['PlayerLine', 'Play', 'Player', 'ActSceneLine']


,column,min_len,median_len,mean_len,p95_len,max_len
0,PlayerLine,1,41.0,38.20,51.0,1029
1,Play,6,14.0,13.94,23.0,24
2,Player,3,8.0,8.56,15.0,18
3,ActSceneLine,3,6.0,6.22,7.0,8


Top 10 longest PlayerLine entries:


,Play,Player,ActSceneLine,PlayerLine_len,PlayerLine
41390,Henry VIII,Old Lady,NaN,1029,"Trumpets, sennet, and cornets. Enter two Verge..."
42684,Henry VIII,GRIFFITH,4.2.88,842,"The vision. Enter, solemnly tripping one after..."
34516,Hamlet,HAMLET,NaN,671,"Enter a King and a Queen very lovingly, the Qu..."
43389,Henry VIII,Porter,NaN,480,"Enter trumpets, sounding, then two Aldermen, L..."
31651,Cymbeline,POSTHUMUS LEONATUS,NaN,423,"Solemn music. Enter, as in an apparition, SIC..."
76900,Pericles,GOWER,3.5.113,406,"Enter, PERICLES and SIMONIDES at one door, wit..."
96542,Titus Andronicus,Captain,1.1.69,336,Drums and trumpets sounded. Enter MARTIUS and ...
66857,Merry Wives of Windsor,MISTRESS QUICKLY,5.5.103,304,During this song they pinch FALSTAFF. DOCTOR C...
31481,Cymbeline,POSTHUMUS LEONATUS,NaN,301,"Enter, from one side, LUCIUS, IACHIMO, and th..."
43070,Henry VIII,KING HENRY VIII,NaN,285,"Enter Chancellor, places himself at the upper ..."


In [7]:
# Play-level and speaker-level aggregates
if 'Play' in df.columns:
    play_counts = df.groupby('Play', dropna=False).size().sort_values(ascending=False).rename('row_count')
    print('Top 15 plays by row count:')
    display(play_counts.head(15).to_frame())

if {'Play', 'Player'}.issubset(df.columns):
    speaker_counts = (
        df.groupby(['Play', 'Player'], dropna=False)
          .size()
          .rename('line_count')
          .reset_index()
          .sort_values(['Play', 'line_count'], ascending=[True, False])
    )

    print('Top 5 speakers within each play (sampled plays):')
    sampled_plays = speaker_counts['Play'].dropna().drop_duplicates().head(5)
    top_speakers_sample = (
        speaker_counts[speaker_counts['Play'].isin(sampled_plays)]
        .groupby('Play', group_keys=False)
        .head(5)
    )
    display(top_speakers_sample)

Top 15 plays by row count:


,row_count
Play,
Hamlet,4244
Coriolanus,3992
Cymbeline,3958
Richard III,3941
Antony and Cleopatra,3862
King Lear,3766
Othello,3762
Troilus and Cressida,3711
A Winters Tale,3489


Top 5 speakers within each play (sampled plays):


,Play,Player,line_count
8,A Comedy of Errors,DROMIO OF SYRACUSE,323
16,A Comedy of Errors,OF SYRACUSE,292
0,A Comedy of Errors,ADRIANA,284
15,A Comedy of Errors,OF EPHESUS,221
7,A Comedy of Errors,DROMIO OF EPHESUS,191
48,A Midsummer nights dream,THESEUS,252
29,A Midsummer nights dream,HELENA,237
38,A Midsummer nights dream,OBERON,234
41,A Midsummer nights dream,PUCK,233
22,A Midsummer nights dream,BOTTOM,220


## Minimal Clean Dataframe For Training

This section applies the required cleaning:
- drop rows where PlayerLinenumber is null
- drop rows where ActSceneLine is null
- keep Dataline, Play, ActSceneLine, PlayerLine

In [14]:
# Required minimal cleaning
required_cols = ['Dataline', 'Play', 'ActSceneLine', 'PlayerLine', 'PlayerLinenumber']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing expected columns: {missing_required}')

before_rows = len(df)

clean_df = (
    df[df['PlayerLinenumber'].notna() & df['ActSceneLine'].notna()]
      .loc[:, ['Dataline', 'Play', 'ActSceneLine', 'PlayerLine']]
      .copy()
      .reset_index(drop=True)
)

after_rows = len(clean_df)
print(f'Rows before filter: {before_rows}')
print(f'Rows after filter:  {after_rows}')
print(f'Rows removed:        {before_rows - after_rows}')

null_check = clean_df.isna().sum().rename('null_count').to_frame()
display(null_check)
display(clean_df.head(10))

output_path = Path('Shakespeare_data_clean_minimal.csv')
if not output_path.parent.exists():
    output_path = Path('dataset') / 'Shakespeare_data_clean_minimal.csv'

clean_df.to_csv(output_path, index=False)
print(f'Saved cleaned file to: {output_path.resolve()}')

Rows before filter: 111396
Rows after filter:  105153
Rows removed:        6243


,null_count
Dataline,0
Play,0
ActSceneLine,0
PlayerLine,0


,Dataline,Play,ActSceneLine,PlayerLine
0,4,Henry IV,1.1.1,"So shaken as we are, so wan with care,"
1,5,Henry IV,1.1.2,"Find we a time for frighted peace to pant,"
2,6,Henry IV,1.1.3,And breathe short-winded accents of new broils
3,7,Henry IV,1.1.4,To be commenced in strands afar remote.
4,8,Henry IV,1.1.5,No more the thirsty entrance of this soil
5,9,Henry IV,1.1.6,Shall daub her lips with her own children's bl...
6,10,Henry IV,1.1.7,"Nor more shall trenching war channel her fields,"
7,11,Henry IV,1.1.8,Nor bruise her flowerets with the armed hoofs
8,12,Henry IV,1.1.9,"Of hostile paces: those opposed eyes,"
9,13,Henry IV,1.1.10,"Which, like the meteors of a troubled heaven,"


Saved cleaned file to: /home/pdutta/bart-trans/dataset/Shakespeare_data_clean_minimal.csv


## What Else To Consider For BART Training Data

Recommended next cleaning steps for text-style transfer:

1. Remove non-dialogue text:
- Filter out lines that look like scene headers or stage directions (for example lines starting with ACT, SCENE, Enter, Exit).

2. Normalize text consistently:
- Strip extra spaces, normalize repeated punctuation, and standardize apostrophes/quotes.

3. Remove low-information/noisy lines:
- Drop extremely short lines (for example length < 5) and rows that become empty after stripping.

4. De-duplicate:
- Remove exact duplicates of Play + PlayerLine to reduce overfitting to repeated lines.

5. Build leakage-safe splits:
- Split train/validation/test by play (or at least by scene) so very similar lines do not leak across splits.

6. Create source-target pairs explicitly:
- For text-to-Shakespeare training, you need aligned pairs: modern_sentence -> shakespeare_sentence.
- If you only have Shakespeare text, consider a synthetic data strategy (back-translation/paraphrasing) to create the modern side.

## Training-Ready Cleaning (v2)

This section builds on the minimal cleaned dataframe and applies stronger preprocessing for style-transfer model training:

- keep dialogue-like lines
- normalize text
- remove noisy/short rows
- deduplicate
- create leakage-safe splits by play

In [15]:
import re
import hashlib

# Start from the minimal clean dataframe if available; otherwise rebuild quickly.
if 'clean_df' not in globals():
    clean_df = (
        df[df['PlayerLinenumber'].notna()]
          .loc[:, ['Dataline', 'Play', 'ActSceneLine', 'PlayerLine']]
          .copy()
          .reset_index(drop=True)
    )

train_df = clean_df.copy()
start_rows = len(train_df)

# 1) Remove obvious non-dialogue lines.
non_dialogue_pattern = re.compile(r'^(ACT\b|SCENE\b|Enter\b|Exit\b|Exeunt\b|Flourish\b|[\[(].*[\])])', re.IGNORECASE)
train_df = train_df[~train_df['PlayerLine'].astype(str).str.strip().str.match(non_dialogue_pattern)]

# 2) Normalize text.
def normalize_text(text: str) -> str:
    t = str(text)
    t = t.replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

train_df['PlayerLine'] = train_df['PlayerLine'].astype(str).map(normalize_text)

# 3) Remove empty/very short lines.
train_df = train_df[train_df['PlayerLine'].str.len() >= 5]

# 4) Drop exact duplicates to reduce repeated memorization.
before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=['Play', 'ActSceneLine', 'PlayerLine']).reset_index(drop=True)

print(f'Start rows (minimal clean): {start_rows}')
print(f'After dialogue/noise filtering: {before_dedup}')
print(f'After deduplication: {len(train_df)}')

display(train_df.head(10))

df_stats = {
    'start_rows': start_rows,
    'after_filter_rows': before_dedup,
    'after_dedup_rows': len(train_df),
    'rows_removed_total': start_rows - len(train_df)
}
print(df_stats)

Start rows (minimal clean): 105153
After dialogue/noise filtering: 104048
After deduplication: 104048


,Dataline,Play,ActSceneLine,PlayerLine
0,4,Henry IV,1.1.1,"So shaken as we are, so wan with care,"
1,5,Henry IV,1.1.2,"Find we a time for frighted peace to pant,"
2,6,Henry IV,1.1.3,And breathe short-winded accents of new broils
3,7,Henry IV,1.1.4,To be commenced in strands afar remote.
4,8,Henry IV,1.1.5,No more the thirsty entrance of this soil
5,9,Henry IV,1.1.6,Shall daub her lips with her own children's bl...
6,10,Henry IV,1.1.7,"Nor more shall trenching war channel her fields,"
7,11,Henry IV,1.1.8,Nor bruise her flowerets with the armed hoofs
8,12,Henry IV,1.1.9,"Of hostile paces: those opposed eyes,"
9,13,Henry IV,1.1.10,"Which, like the meteors of a troubled heaven,"


{'start_rows': 105153, 'after_filter_rows': 104048, 'after_dedup_rows': 104048, 'rows_removed_total': 1105}


In [16]:
# 5) Leakage-safe split by Play using deterministic hashing.

def assign_split(play_value: str) -> str:
    key = str(play_value).strip().lower().encode('utf-8')
    bucket = int(hashlib.md5(key).hexdigest(), 16) % 100
    if bucket < 80:
        return 'train'
    if bucket < 90:
        return 'val'
    return 'test'

train_df = train_df.copy()
train_df['split'] = train_df['Play'].map(assign_split)

split_counts = train_df['split'].value_counts().rename('row_count').to_frame()
play_split_table = (
    train_df.groupby(['split', 'Play'])
            .size()
            .rename('rows')
            .reset_index()
            .sort_values(['split', 'rows'], ascending=[True, False])
)

base_out = Path('dataset') if Path('dataset').exists() else Path('.')
full_out = base_out / 'Shakespeare_data_clean_v2.csv'
train_out = base_out / 'Shakespeare_data_clean_v2_train.csv'
val_out = base_out / 'Shakespeare_data_clean_v2_val.csv'
test_out = base_out / 'Shakespeare_data_clean_v2_test.csv'

train_df.to_csv(full_out, index=False)
train_df[train_df['split'] == 'train'].to_csv(train_out, index=False)
train_df[train_df['split'] == 'val'].to_csv(val_out, index=False)
train_df[train_df['split'] == 'test'].to_csv(test_out, index=False)

print('Saved files:')
print(full_out.resolve())
print(train_out.resolve())
print(val_out.resolve())
print(test_out.resolve())

display(split_counts)
display(play_split_table.head(15))

Saved files:
/home/pdutta/bart-trans/dataset/Shakespeare_data_clean_v2.csv
/home/pdutta/bart-trans/dataset/Shakespeare_data_clean_v2_train.csv
/home/pdutta/bart-trans/dataset/Shakespeare_data_clean_v2_val.csv
/home/pdutta/bart-trans/dataset/Shakespeare_data_clean_v2_test.csv


,row_count
split,
train,80062
test,17887
val,6099


,split,Play,rows
1,test,Henry IV,3025
0,test,As you like it,2666
2,test,Merchant of Venice,2653
3,test,Taming of the Shrew,2616
5,test,Timon of Athens,2461
4,test,The Tempest,2256
6,test,Two Gentlemen of Verona,2210
14,train,Hamlet,3963
12,train,Coriolanus,3743
13,train,Cymbeline,3708


## Next analysis ideas

- Compare average line length by play to identify stylistic differences.
- Track speaker dominance across acts/scenes when `ActSceneLine` is parseable.
- Build a cleaned training table for downstream NLP tasks (speaker prediction, style transfer, summarization).